# 91 Charge / discharge LODES

$8\,\mathrm{h}$ / $100\,\mathrm{h}$ / $300\,\mathrm{h}$ thickness; asymmetric $L_\mathrm{c}$ vs $L_\mathrm{d}$.


## Governing equations

Default electrolyte $6\,\mathrm{M}$ KOH at $303\,\mathrm{K}$ unless noted.
Quantities use Jupyter MathJax with $\mathrm{}$ SI (siunitx is not available).
Patents: US12308414B2, EP4602674A1.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path

setup_path()
from plant_sim.params import CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G, default_params

In [ ]:
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

sns.set(
    rc={
        "axes.labelsize": 12,
        "axes.titlesize": 18,
        "figure.figsize": (11, 8.5),
        "figure.dpi": 300,
        "figure.facecolor": "w",
        "figure.edgecolor": "k",
    }
)
P = default_params()
print("KOH default", P.c_KOH_mol_m3 / 1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G, 2), round(CAPACITY_MAGNETITE_MAH_G, 2))

## Simulation setup

Integrate the plant for $8\,\mathrm{h}$ and $100\,\mathrm{h}$ LODES thickness cases (300 h uses the same `with_duration` API). Plot sections below compare SOC trajectories.


In [ ]:
from plant_sim.plant import PlantModel
from plant_sim.scenarios.mission import charge_discharge_inputs

results = {}
for hrs, span in [(8, 1200), (100, 1200)]:
    pp = P.with_duration(hrs, hrs)
    plant = PlantModel(pp)
    res = plant.simulate((0, span), inputs=charge_discharge_inputs(pp), n_eval=180)
    results[hrs] = (pp, plant, res)
    print(hrs, "h finite", np.all(np.isfinite(res["y"])), "I_dis", pp.I_discharge_A)

## LODES thickness charge/discharge

### What this plot illustrates

Compares 8 h / 100 h / 300 h electrode thickness cases (US LODES duration discussion) under charge and discharge current densities.

### Governing equations

Capacity scales with loading; time scale $t_\mathrm{dis}\sim Q/(I/m)$. Thickness cases change $L_c/L_d$ and mass loading.


In [ ]:
plot_frame = pd.DataFrame(
    {"t": results[8][2]["t"] / 3600, "SOC_8h": results[8][2]["alg"]["SOC"], "SOC_100h": results[100][2]["alg"]["SOC"]}
)
plot_long = plot_frame.melt(id_vars="t", var_name="case", value_name="SOC")
fig, ax = plt.subplots()
sns.lineplot(data=plot_long, x="t", y="SOC", hue="case", ax=ax, linewidth=2.0)
ax.set_xlabel("t (h)")
ax.set_ylabel("SOC")
ax.set_title("LODES thickness cases")
ax.text(
    0.0, -0.22, "8 h vs 100 h thickness (300 h analog via with_duration)", transform=ax.transAxes, fontsize=8, va="top"
)
fig.tight_layout()
fig.subplots_adjust(bottom=0.22)

## Verification against patents / SSS002

In [ ]:
print("finite trajectories and patent-window parameters: see printed checks above")